# RAG 问答 Notebook

上传 10-K PDF → 提问 → 获得 cited 回答

**使用方式**：按顺序执行每个 Cell 即可

In [1]:
# === 1. 初始化 ===
import sys, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from config import BASE_DIR
from core.vector_store import vector_store
from services.document_service import process_upload
from services.chat_service import ask_question

# 检查 ChromaDB，为空则自动加载 10-K PDF
res = vector_store.vectorstore.get(include=["metadatas"])
if not res.get("metadatas"):
    print("ChromaDB 为空，正在加载 10-K PDF...")
    for pdf in sorted((BASE_DIR / "10-k").glob("*.pdf")):
        print(f"  加载: {pdf.name}")
        process_upload(str(pdf), pdf.name)
    print("加载完成！")
else:
    print(f"ChromaDB 已有 {len(res['metadatas'])} 条记录")

d:\python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9694.87it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[REGISTRY] Restored 0 documents from ChromaDB
ChromaDB 为空，正在加载 10-K PDF...
  加载: apple-2025.pdf
加载完成！


In [2]:
# === 2. 单次问答演示 ===
question = "What was Apple's total net sales in fiscal year 2025?"

print(f"Q: {question}\n")
t0 = time.time()
result = ask_question(question)
elapsed = time.time() - t0

print(f"A: {result['answer']}")
print(f"\n--- 引用 ({len(result.get('citations', []))} 条) ---")
for c in result.get("citations", []):
    print(f"  [{c.get('company_name', '?')}] {c.get('section', '?')} | Page {c.get('page_no', '?')}")
if result.get("chart_data"):
    print(f"\n图表数据: {list(result['chart_data'].keys())}")
print(f"\n耗时: {elapsed:.1f}s | 步骤: {result.get('workflow_steps', [])}")

Q: What was Apple's total net sales in fiscal year 2025?


[QUERY_REWRITER] Original: What was Apple's total net sales in fiscal year 2025?
[QUERY_REWRITER] Rewritten: What was Apple's total net sales, also referred to as total revenue, net revenue, or total sales, for the fiscal year ended in 2025, including breakdowns by products and services, reported in millions or billions of United States dollars?
[METADATA_EXTRACTOR] companies=['apple'], year=2025, quarter=

[RETRIEVER] Query: What was Apple's total net sales, also referred to as total revenue, net revenue, or total sales, fo
[RETRIEVER] Companies: ['apple']
[RETRIEVER] Filters: {'year': '2025'}
[RETRIEVER] Needs financials: True | Needs risk: False
[RETRIEVER] Hybrid reranked: 10 docs (alpha=0.7)
  Doc 0: company=apple section=item_8_financials page=32 type=text text=Apple Inc.
CONSOLIDATED STATEMENTS OF OPERATIONS
(In millions, except number of ...
  Doc 1: company=apple section=item_8_financials page=39 type=text text=The fol